<center>




# Bronze to Silver

</center>

# **Purpose**

#### The text book example of a data lakehouse has three layers, where data is moved between. Sometimes it is named Bronze, Silver and Gold. The medallion-lakehouse uses the classic Bronze, Silver and Gold layer names.

##### Bronze: Holds the data in original format if possible. Relation sources are stored in parquet. Delta is not needed here, as the layer is not intended for re-writing or querying.
##### Silver: Aligned clean data, all stored in Delta tables.
##### Gold: Data for serving, business logic is applied, star schemas are defined etc. 

#### Moving data between Bronze and Silver can be trivial and cumbersome, it is also a process easy to describe with meta data. This notebook reads meta data and processes data from it. We are also creating a date dimension from a generator defined in shared functions. 

### **Load utility function**

#### The notebook builds upon functions defined in medallion-lakehouse_functions and medallion-lakehouse_custom_transforms. Calling other notebooks with %run must be defined in top code cell. This step can be omitted having shared functions in a python wheel and importing them as custom library. 

In [ ]:
%run medallion-lakehouse_functions

In [ ]:
%run medallion-lakehouse_custom_transforms


# **Defining settings**
#### Some settings run through the notebook and easy to set once before the job starts.

In [ ]:
destination_lakehouse = 'silver'

date_dimension_start_date = '2022-01-01'
date_dimension_end_date = '2023-12-31'
date_dimension_destination = 'dbo.dim_dates'


# **Reading meta data**

#### We are reading meta data from JSON, it could have been from a database or YAML file as well. The important part is it ends in a common structure. 

In [ ]:
meta_data = read_meta_from_json("/lakehouse/default/Files/meta/meta_data.json")

# **Print out meta data** (Not needed)

#### Handy for debugging, but not needed when running the notebook.

In [ ]:
meta_data

# **Processing**

#### First the data dimension is created.
#### We then loops over the meta data for processing the tables.

In [ ]:
create_or_update_date_dimension(
    start_date=date_dimension_start_date,
    end_date=date_dimension_end_date,
    lakehouse=destination_lakehouse,
    destination=date_dimension_destination,
)

for entity in meta_data:
    meta_entity = meta_data[entity]
    print(f"processing {entity} from {meta_entity['source']}")
    df = meta_entity['reader_function']()

    custom_name = meta_entity.get('custom_transform')
    if custom_name:
        print(f"  applying custom_transform: {custom_name}")
        df = run_custom_transform(custom_name, df, meta_entity)
    else:
        print("  default flow: dropDuplicates + fill nulls")
        df = df.dropDuplicates()
        df = df.na.fill(0)
        df = df.na.fill("N/A")

    write_to_delta_overwrite(
        df=df,
        lakehouse=destination_lakehouse,
        destination=meta_entity['destination'],
        projectedColumns=meta_entity['projection'],
    )
